# 🥚 Pelatihan & Analisis Arsitektur YOLOv8-seg (Instance Segmentation) - Cacat Cangkang Telur
Notebook ini melatih model **YOLOv8-seg** untuk melakukan **segmentasi piksel cacat retak (*cracked*) dan noda kotoran (*dirt*)** pada cangkang telur ayam sesuai acuan standar SNI 3926:2023, sekaligus menghasilkan **tabel spesifikasi arsitektur, rincian layer-per-layer konvolusi, struktur pohon blok, serta analisis BatchNorm2d, SiLU, dan ReLU** untuk laporan.

### 📊 Distribusi & Split Dataset (70% Train / 20% Valid / 10% Test):
- **Total Dataset:** 892 gambar (1.166 polygon mask cacat)
  - **Train (70%):** 624 gambar (207 cracked, 225 dirt, 192 good)
  - **Valid (20%):** 178 gambar (59 cracked, 64 dirt, 55 good)
  - **Test (10%):** 90 gambar (30 cracked, 32 dirt, 28 good)

### 🏷️ Pembagian Kelas Mutu SNI:
1. `cracked` (296 gambar, 330 polygon instances): Cacat kritis retak kerabang (Reject / Mutu III).
2. `dirt` (321 gambar, 836 polygon instances): Noda kotoran / feses permukaan (Mutu II / III).
3. `good` (275 gambar, label kosong): Sampel negatif background cangkang bersih mulus (Mutu I).

### ⚙️ Konfigurasi Pelatihan & Analisis:
- **Arsitektur:** `yolov8n-seg.pt` (Instance Segmentation — 23 blok utama: Backbone, Neck FPN+PAN, Head Segment + ProtoNet)
- **Epochs:** 80 (Patience Early Stopping: 15) | **Input Resolution:** 640×640
- **Output Analisis Model:** Tabel 1 (Ringkasan), Tabel 2 (Kelas), Tabel 3 (Jenis Modul), Tabel 4 (Detail Blok Utama), Tabel 4B (Semua Sub-Layer + Ukuran Tensor), Tabel 4C (Ringkasan per Tipe), Tabel 4D (Verifikasi Rumus Konvolusi), Struktur Pohon (`struktur_model.txt`), serta Grafik BatchNorm2d, SiLU, dan ReLU.

In [ ]:
# 1. Cek Akselerasi GPU Colab & Install Ultralytics
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# 2. Persiapan & Ekstraksi Dataset Segmentasi
import os
import zipfile
import subprocess

extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)

# Cari file zip di folder /content
zip_candidates = [f for f in os.listdir('/content') if f.endswith('.zip') and not f.startswith('.')]

if not zip_candidates:
    print('ℹ️ Belum ada file .zip di /content.')
    print("👉 Silakan upload file 'Egg_Dataset_Clean_70_20_10.zip' ke panel Files (kiri Colab).")
    print('👉 PERHATIKAN: Tunggu sampai lingkaran orange progress upload di pojok kiri bawah selesai penuh!')
else:
    zip_path = os.path.join('/content', zip_candidates[0])
    file_size_bytes = os.path.getsize(zip_path)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f'📦 File terdeteksi : {zip_path}')
    print(f'💾 Ukuran file     : {file_size_mb:.2f} MB ({file_size_bytes:,} bytes)')
    
    # Validasi: Ukuran asli dataset adalah ~23.5 MB
    if file_size_mb < 5.0:
        print('\n⚠️ PERINGATAN: Ukuran file zip terlalu kecil (< 5 MB)!')
        print('Penyebab error: File zip belum selesai di-upload 100% atau koneksi sempat terputus.')
        print('Cara mengatasi:')
        print('  1. Klik kanan file zip di panel Files kiri Colab -> Delete file.')
        print('  2. Upload ulang file Egg_Dataset_Clean_70_20_10.zip.')
        print('  3. Tunggu lingkaran progress di kiri bawah sampai 100% selesai, baru jalankan cell ini!')
    else:
        print(f'\n🚀 Mengekstrak ke {extract_dir}...')
        res = subprocess.run(['unzip', '-q', '-o', zip_path, '-d', extract_dir], capture_output=True, text=True)
        if res.returncode == 0:
            print('✅ Ekstraksi dataset berhasil!')
        else:
            print(f'Peringatan command unzip: {res.stderr}. Mencoba modul zipfile...')
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print('✅ Ekstraksi via zipfile berhasil!')
        
        print(f'📂 Isi folder {extract_dir}: {os.listdir(extract_dir)}')

In [ ]:
# 3. Konfigurasi data.yaml (Path Dataset Colab)
import os
import yaml

found_yaml = None
for root, dirs, files in os.walk('/content'):
    if 'data.yaml' in files:
        found_yaml = os.path.join(root, 'data.yaml')
        break

if found_yaml:
    with open(found_yaml, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['path'] = os.path.dirname(found_yaml)
    with open(found_yaml, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    data_yaml_path = found_yaml
    print(f'✅ Konfigurasi data.yaml ditemukan di: {data_yaml_path}')
    print(yaml.dump(cfg))
else:
    data_yaml_path = '/content/dataset/data.yaml'
    default_cfg = {
        'path': '/content/dataset',
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 2,
        'names': ['cracked', 'dirt']
    }
    os.makedirs('/content/dataset', exist_ok=True)
    with open(data_yaml_path, 'w') as f:
        yaml.dump(default_cfg, f, default_flow_style=False)
    print(f'✅ data.yaml baru berhasil dibuat di: {data_yaml_path}')
    print(yaml.dump(default_cfg))

In [ ]:
# 4. Mulai Pelatihan Model YOLOv8-seg (Instance Segmentation)
from ultralytics import YOLO
import torch

train_device = 0 if torch.cuda.is_available() else 'cpu'
batch_size = 16 if torch.cuda.is_available() else 8
epochs_num = 80

print(f'🚀 Memulai pelatihan YOLOv8-seg pada perangkat: {train_device} (Batch: {batch_size}, Epochs: {epochs_num})...')

# Memuat arsitektur pre-trained YOLOv8 Nano Segmentation
model = YOLO('yolov8n-seg.pt')

results = model.train(
    data=data_yaml_path,
    epochs=epochs_num,
    imgsz=640,
    batch=batch_size,
    device=train_device,
    project='/content/runs',
    name='egg_defect_yolov8_seg',
    exist_ok=True,
    patience=15,
    save=True,
    plots=True,
    verbose=True,
    # Parameter augmentasi terarah khusus retakan rambut & cangkang telur:
    mosaic=0.5,
    close_mosaic=10,
    hsv_v=0.4,
    hsv_s=0.4,
    fliplr=0.5,
    flipud=0.5
)

In [ ]:
# 5. Evaluasi Performa Model (Box mAP & Mask mAP)
metrics = model.val(
    data=data_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='eval_test_seg'
)

print('\n' + '='*55)
print('📊 HASIL EVALUASI SEGMENTASI (TEST SET):')
print('='*55)
print(f'📦 Bounding Box mAP50        : {metrics.box.map50:.4f}')
print(f'📦 Bounding Box mAP50-95     : {metrics.box.map:.4f}')
print(f'🎭 Segmentation Mask mAP50   : {metrics.seg.map50:.4f}')
print(f'🎭 Segmentation Mask mAP50-95: {metrics.seg.map:.4f}')
print('='*55)

In [ ]:
# 6. Visualisasi Hasil Segmentasi (Masking Cacat Cangkang)
import os
import glob
from IPython.display import Image, display

val_preds = glob.glob('/content/runs/egg_defect_yolov8_seg/val_batch*_pred.jpg')
charts = [
    '/content/runs/egg_defect_yolov8_seg/results.png',
    '/content/runs/egg_defect_yolov8_seg/confusion_matrix.png'
]

for path in charts + val_preds[:2]:
    if os.path.exists(path):
        print(f'\n=== {os.path.basename(path)} ===')
        display(Image(path, width=720))

---
## 🔍 Bagian 7: Analisis Arsitektur Model & Rincian Layer-per-Layer (`YOLOv8n-seg`)
Bagian ini membedah struktur internal jaringan saraf tiruan **YOLOv8-seg** (sebelum proses *fuse* `Conv2d + BatchNorm2d`) ke dalam bentuk tabel dan grafik:
- **Tabel 1:** Ringkasan spesifikasi model (ukuran file, resolusi input, jumlah blok utama, total sub-modul, total parameter, GFLOPs).
- **Tabel 2:** Daftar kelas target segmentasi.
- **Tabel 3:** Rekapitulasi jenis modul penyusun arsitektur.
- **Tabel 4 & Grafik:** Detail 23 blok utama (Layer 0 s/d Layer 22: Backbone, Neck FPN+PAN, Head Segment + ProtoNet).
- **Tabel 4B, 4C, 4D & Grafik:** Seluruh sub-layer (`Conv2d`, `BatchNorm2d`, `SiLU`, `ConvTranspose2d`, `Upsample`, `Concat`, `MaxPool2d`), jumlah *feature maps*, kernel, stride, padding, dimensi tensor masuk/keluar ($N \times C \times H \times W$), serta verifikasi matematis rumus konvolusi.
- **Struktur Pohon Blok (`struktur_model.txt`):** Hirarki layer di dalam setiap blok beserta jumlah parameter masing-masing.

In [ ]:
# 7.1 Ekstraksi Info Model YOLOv8-seg (Tabel 1, 2, 3, 4 & Grafik Parameter per Blok)
import os
import copy
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops

# Gunakan bobot terbaik hasil training jika ada; jika belum training, gunakan yolov8n-seg.pt
best_weight_path = '/content/runs/egg_defect_yolov8_seg/weights/best.pt'
active_model_path = best_weight_path if os.path.exists(best_weight_path) else 'yolov8n-seg.pt'

# Muat salinan model BARU (unfused) agar BatchNorm2d tetap utuh dan tidak menyatu ke Conv2d
yolo_inspect = YOLO(active_model_path)
net = copy.deepcopy(yolo_inspect.model).cpu().eval()

file_size_mb = round(os.path.getsize(active_model_path) / (1024 * 1024), 2) if os.path.exists(active_model_path) else 0.0
total_params = sum(p.numel() for p in net.parameters())
total_modules = sum(1 for _ in net.modules())
gflops = round(get_flops(net, imgsz=640), 2)
class_names = yolo_inspect.names  # {0: 'cracked', 1: 'dirt'}

# --- TABEL 1: Ringkasan Model ---
tabel_ringkasan = pd.DataFrame([
    {'Item': 'Nama file model', 'Nilai': os.path.basename(active_model_path)},
    {'Item': 'Arsitektur dasar', 'Nilai': 'YOLOv8n-seg (Instance Segmentation)'},
    {'Item': 'Ukuran file (MB)', 'Nilai': file_size_mb},
    {'Item': 'Tugas', 'Nilai': str(yolo_inspect.task)},
    {'Item': 'Ukuran input', 'Nilai': '640 x 640'},
    {'Item': 'Jumlah kelas aktif', 'Nilai': len(class_names)},
    {'Item': 'Jumlah blok utama', 'Nilai': len(net.model)},
    {'Item': 'Jumlah modul total', 'Nilai': total_modules},
    {'Item': 'Total parameter', 'Nilai': f'{total_params:,}'},
    {'Item': 'GFLOPs', 'Nilai': gflops}
])
print('TABEL 1. Ringkasan model')
display(tabel_ringkasan)

# --- TABEL 2: Kelas ---
tabel_kelas = pd.DataFrame({
    'ID': list(class_names.keys()),
    'Nama kelas': list(class_names.values()),
    'Keterangan': ['Cacat kritis retak kerabang (Mutu III / Reject)', 'Noda kotoran / feses permukaan (Mutu II / III)'][:len(class_names)]
})
print('\nTABEL 2. Kelas')
display(tabel_kelas)

# --- TABEL 3: Jenis Modul ---
modul_counts = {}
for m in net.modules():
    t_name = type(m).__name__
    modul_counts[t_name] = modul_counts.get(t_name, 0) + 1

tabel_modul = pd.DataFrame([
    {'Jenis modul': k, 'Jumlah': v} for k, v in modul_counts.items()
]).sort_values(by='Jumlah', ascending=False).reset_index(drop=True)
print('\nTABEL 3. Jenis modul penyusun model')
display(tabel_modul)

# --- TABEL 4: Detail per Blok Utama (Layer 0 s/d Layer 22) ---
rows_blok = []
for idx, block in enumerate(net.model):
    b_params = sum(p.numel() for p in block.parameters())
    n_conv = sum(1 for m in block.modules() if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)))
    n_bn = sum(1 for m in block.modules() if isinstance(m, nn.BatchNorm2d))
    n_silu = sum(1 for m in block.modules() if isinstance(m, nn.SiLU))
    dari_val = str(getattr(block, 'f', -1))
    rows_blok.append({
        'No': idx,
        'Dari': dari_val,
        'Tipe': type(block).__name__,
        'Parameter': b_params,
        'Conv2d': n_conv,
        'BatchNorm2d': n_bn,
        'Aktivasi': f'SiLU x{n_silu}' if n_silu > 0 else '-',
        'Persen parameter': round((b_params / total_params) * 100, 2)
    })

tabel_layer = pd.DataFrame(rows_blok)
pd.set_option('display.max_rows', None)
print('\nTABEL 4. Detail per layer utama (Blok 0 - 22)')
display(tabel_layer)

# Simpan ke CSV
tabel_ringkasan.to_csv('/content/info_model_ringkasan.csv', index=False)
tabel_kelas.to_csv('/content/info_model_kelas.csv', index=False)
tabel_modul.to_csv('/content/info_model_modul.csv', index=False)
tabel_layer.to_csv('/content/info_model_layer.csv', index=False)
print('Tersimpan: info_model_ringkasan.csv, info_model_kelas.csv, info_model_modul.csv, info_model_layer.csv')

# --- Grafik Jumlah Parameter per Blok Utama ---
plt.figure(figsize=(12, 4.5))
labels_blok = [f"{r['No']}:{r['Tipe']}" for _, r in tabel_layer.iterrows()]
plt.bar(labels_blok, tabel_layer['Parameter'], color='#2563eb')
plt.xticks(rotation=65, ha='right', fontsize=9)
plt.ylabel('Jumlah Parameter')
plt.title('Distribusi Jumlah Parameter per Blok Utama YOLOv8n-seg')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('/content/grafik_parameter_layer.png', dpi=200)
plt.show()

In [ ]:
# 7.2 Rincian Semua Sub-Layer (Conv2d, BatchNorm2d, SiLU, Upsample, Concat, dll.) & Verifikasi Rumus Konvolusi
import math
import copy
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Gunakan salinan model bersih untuk menangkap dimensi tensor masuk & keluar via forward hook
net_hook = copy.deepcopy(yolo_inspect.model).cpu().eval()

target_types = (
    nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d,
    nn.SiLU, nn.ReLU, nn.Upsample, nn.MaxPool2d
)

records = []
hooks = []

def fmt_tuple(val):
    if isinstance(val, tuple):
        return 'x'.join(str(x) for x in val)
    return str(val)

for name, mod in net_hook.named_modules():
    if isinstance(mod, target_types) or type(mod).__name__ == 'Concat':
        def make_hook(layer_name, layer_mod):
            def hook_fn(m, inp, out):
                # Ekstrak shape masuk
                if isinstance(inp, (tuple, list)) and len(inp) > 0:
                    first_in = inp[0]
                    if isinstance(first_in, (tuple, list)):
                        in_shapes = [' x '.join(str(d) for d in t.shape) for t in first_in if hasattr(t, 'shape')]
                        in_shape_str = ' + '.join(in_shapes)
                        in_c = sum(t.shape[1] for t in first_in if hasattr(t, 'shape') and len(t.shape) >= 2)
                        h_in = first_in[0].shape[2] if len(first_in[0].shape) >= 4 else None
                    elif hasattr(first_in, 'shape'):
                        in_shape_str = ' x '.join(str(d) for d in first_in.shape)
                        in_c = first_in.shape[1] if len(first_in.shape) >= 2 else '-'
                        h_in = first_in.shape[2] if len(first_in.shape) >= 4 else None
                    else:
                        in_shape_str, in_c, h_in = '-', '-', None
                else:
                    in_shape_str, in_c, h_in = '-', '-', None

                # Ekstrak shape keluar
                if hasattr(out, 'shape'):
                    out_shape_str = ' x '.join(str(d) for d in out.shape)
                    out_c = out.shape[1] if len(out.shape) >= 2 else '-'
                    h_out = out.shape[2] if len(out.shape) >= 4 else None
                else:
                    out_shape_str, out_c, h_out = '-', '-', None

                t_name = type(layer_mod).__name__
                p_count = sum(p.numel() for p in layer_mod.parameters(recurse=False))

                kernel = fmt_tuple(layer_mod.kernel_size) if hasattr(layer_mod, 'kernel_size') else '-'
                stride = fmt_tuple(layer_mod.stride) if hasattr(layer_mod, 'stride') else '-'
                padding = fmt_tuple(layer_mod.padding) if hasattr(layer_mod, 'padding') else '-'
                groups = str(layer_mod.groups) if hasattr(layer_mod, 'groups') else '-'
                w_shape = ' x '.join(str(d) for d in layer_mod.weight.shape) if hasattr(layer_mod, 'weight') and layer_mod.weight is not None else '-'

                records.append({
                    'No': len(records) + 1,
                    'Nama layer': layer_name,
                    'Tipe': t_name,
                    'Kanal masuk': in_c,
                    'Feature maps (kanal keluar)': out_c,
                    'Kernel': kernel,
                    'Stride': stride,
                    'Padding': padding,
                    'Groups': groups,
                    'Bentuk bobot': w_shape,
                    'Bentuk masuk (N x C x H x W)': in_shape_str,
                    'Bentuk keluar (N x C x H x W)': out_shape_str,
                    'Parameter': p_count,
                    '_h_in': h_in,
                    '_h_out': h_out,
                    '_mod': layer_mod
                })
            return hook_fn
        hooks.append(mod.register_forward_hook(make_hook(name, mod)))

# Jalankan 1x forward pass dengan citra dummy 640x640
with torch.no_grad():
    dummy_img = torch.zeros(1, 3, 640, 640)
    _ = net_hook(dummy_img)

for h in hooks:
    h.remove()

df_all = pd.DataFrame(records)
kolom_tampil = [
    'No', 'Nama layer', 'Tipe', 'Kanal masuk', 'Feature maps (kanal keluar)',
    'Kernel', 'Stride', 'Padding', 'Groups', 'Bentuk bobot',
    'Bentuk masuk (N x C x H x W)', 'Bentuk keluar (N x C x H x W)', 'Parameter'
]
tabel_semua = df_all[kolom_tampil].copy()
tabel_semua['Persen parameter'] = (tabel_semua['Parameter'] / tabel_semua['Parameter'].sum() * 100).round(3)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(f'TABEL 4B. Semua layer model: {len(tabel_semua)} layer')
display(tabel_semua)
print(f"Jumlah parameter dari tabel = {tabel_semua['Parameter'].sum():,} | total dari model = {total_params:,}")
print("Tanda '-' berarti atribut itu tidak ada di layer tersebut (misalnya SiLU tidak punya kernel, stride, padding).")

# --- TABEL 4C: Ringkasan per Tipe Layer ---
tabel_per_tipe = tabel_semua.groupby('Tipe').agg(
    Jumlah=('No', 'count'),
    Parameter=('Parameter', 'sum')
).sort_values('Parameter', ascending=False).reset_index()
print('\nTABEL 4C. Ringkasan per tipe layer')
display(tabel_per_tipe)

# --- TABEL 4D: Khusus Konvolusi (Conv2d) + Pengecekan Rumus Ukuran & Parameter ---
conv_rows = []
for _, r in df_all[df_all['Tipe'] == 'Conv2d'].iterrows():
    m = r['_mod']
    h_in, h_out = r['_h_in'], r['_h_out']
    k = m.kernel_size[0]
    s = m.stride[0]
    p = m.padding[0]
    g = m.groups
    c_in, c_out = m.in_channels, m.out_channels
    h_calc = math.floor((h_in + 2 * p - k) / s) + 1 if h_in is not None else None
    p_calc = c_out * (c_in // g) * k * k + (c_out if m.bias is not None else 0)
    conv_rows.append({
        'No': r['No'],
        'Nama layer': r['Nama layer'],
        'C_in': c_in,
        'Feature maps (C_out)': c_out,
        'Kernel': f'{k}x{k}',
        'Stride': s,
        'Padding': p,
        'Resolusi masuk': f'{h_in}x{h_in}',
        'Resolusi keluar': f'{h_out}x{h_out}',
        'Ukuran cocok rumus': 'ya' if h_calc == h_out else 'tidak',
        'Parameter': r['Parameter'],
        'Parameter cocok rumus': 'ya' if p_calc == r['Parameter'] else 'tidak'
    })

tabel_konvolusi = pd.DataFrame(conv_rows)
print('\nTABEL 4D. Analisis & Verifikasi Rumus Layer Konvolusi (Conv2d)')
display(tabel_konvolusi)

# Simpan CSV
tabel_semua.to_csv('/content/info_model_semua_layer.csv', index=False)
tabel_per_tipe.to_csv('/content/info_model_per_tipe.csv', index=False)
tabel_konvolusi.to_csv('/content/info_model_konvolusi.csv', index=False)
print('Tersimpan: info_model_semua_layer.csv, info_model_per_tipe.csv, info_model_konvolusi.csv')

# --- Grafik Layer per Tipe ---
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(tabel_per_tipe['Tipe'], tabel_per_tipe['Jumlah'], color='#0284c7', alpha=0.85, label='Jumlah Layer')
ax1.set_ylabel('Jumlah Layer', color='#0284c7')
ax2 = ax1.twinx()
ax2.plot(tabel_per_tipe['Tipe'], tabel_per_tipe['Parameter'], color='#dc2626', marker='o', linewidth=2, label='Total Parameter')
ax2.set_ylabel('Total Parameter', color='#dc2626')
plt.title('Jumlah Layer dan Total Parameter per Tipe Modul (YOLOv8n-seg)')
fig.tight_layout()
plt.savefig('/content/grafik_layer_per_tipe.png', dpi=200)
plt.show()

# --- Grafik Feature Maps vs Resolusi Keluaran Konvolusi ---
res_keluar_num = [int(x.split('x')[0]) for x in tabel_konvolusi['Resolusi keluar']]
fig, ax1 = plt.subplots(figsize=(12, 4.2))
x_idx = range(1, len(tabel_konvolusi) + 1)
ax1.plot(x_idx, tabel_konvolusi['Feature maps (C_out)'], color='#16a34a', marker='o', markersize=4, label='Jumlah Feature Maps (Kanal Keluar)')
ax1.set_xlabel('Urutan Layer Conv2d')
ax1.set_ylabel('Jumlah Feature Maps (Channels)', color='#16a34a')
ax2 = ax1.twinx()
ax2.plot(x_idx, res_keluar_num, color='#ea580c', linestyle='--', marker='s', markersize=4, label='Resolusi Spasial Keluar (px)')
ax2.set_ylabel('Resolusi Spasial (H_out / W_out)', color='#ea580c')
plt.title('Dinamika Feature Maps vs Resolusi Spasial di Sepanjang Layer Konvolusi YOLOv8n-seg')
fig.tight_layout()
plt.savefig('/content/grafik_konvolusi_featuremaps_resolusi.png', dpi=200)
plt.show()

In [ ]:
# 7.3 Tampilan Struktur Pohon Layer-per-Layer per Blok (struktur_model.txt)
# Ubah BLOK_TAMPIL = [0, 1, 2, 22] jika hanya ingin menampilkan blok tertentu; None untuk semua blok (0..22)
BLOK_TAMPIL = None

lines_tree = []
for idx, block in enumerate(net.model):
    if BLOK_TAMPIL is not None and idx not in BLOK_TAMPIL:
        continue
    b_type = type(block).__name__
    lines_tree.append('-' * 70)
    lines_tree.append(f'BLOCK {idx}: {b_type}')
    lines_tree.append('-' * 70)
    for sub_name, sub_mod in block.named_modules():
        own_params = sum(p.numel() for p in sub_mod.parameters(recurse=False))
        sub_type = type(sub_mod).__name__
        if sub_name == '':
            lines_tree.append(f'Layer {idx} [{sub_type}] Parameters: {own_params:,}')
        else:
            depth = sub_name.count('.') + 1
            indent = '    ' * depth
            short_name = sub_name.split('.')[-1]
            lines_tree.append(f'{indent}{short_name} [{sub_type}] Parameters: {own_params:,}')

tree_text = '\n'.join(lines_tree)
with open('/content/struktur_model.txt', 'w') as f:
    f.write(tree_text)

print('✅ Struktur pohon lengkap tersimpan ke: /content/struktur_model.txt')
# Tampilkan pratinjau Blok 0, Blok 1, Blok 2 (C2f), dan Blok 22 (Head Segment)
for preview_idx in [0, 1, 2, 22]:
    blk = net.model[preview_idx]
    print('-' * 70)
    print(f'BLOCK {preview_idx}: {type(blk).__name__}')
    print('-' * 70)
    for s_name, s_mod in blk.named_modules():
        p_own = sum(p.numel() for p in s_mod.parameters(recurse=False))
        if s_name == '':
            print(f'Layer {preview_idx} [{type(s_mod).__name__}] Parameters: {p_own:,}')
        else:
            d = s_name.count('.') + 1
            print(f"{'    ' * d}{s_name.split('.')[-1]} [{type(s_mod).__name__}] Parameters: {p_own:,}")

---
## 📈 Bagian 8: Analisis & Visualisasi BatchNorm2d, Aktivasi SiLU, dan ReLU pada `YOLOv8n-seg`
Bagian ini membuktikan cara kerja internal blok konvolusi YOLOv8-seg (`Conv2d -> BatchNorm2d -> SiLU`):
- **8.1 Batch Normalization dari Nol (NumPy):** Perhitungan $\mu$, $\sigma^2$, normalisasi $\hat{x}$, serta parameter terlatih $\gamma$ dan $\beta$ lengkap dengan tabel statistik dan histogram distribusi nilai.
- **8.2 Grafik Fungsi Aktivasi SiLU vs ReLU & Turunannya:** Kurva fungsi non-linear dan gradien turunan pertama, termasuk titik minimum global SiLU ($\approx -0,2785$ pada $x \approx -1,28$).
- **8.3 Perbandingan Urutan `BatchNorm -> Aktivasi`:** Mengukur efek *sparsity* (nilai nol tepat / *dying ReLU*) antara ReLU vs SiLU.
- **8.4 Inspeksi Langsung Model `YOLOv8n-seg` Cangkang Telur:** Distribusi bobot $\gamma, \beta, \mu, \sigma^2$ hasil training, uji aktivasi layer pertama (`model.model[0]`) pada citra cangkang telur, serta efek penggabungan layer (`model.fuse()`).

In [ ]:
# 8.1 - 8.4 Analisis BatchNorm2d, SiLU, ReLU & Inspeksi Bobot Terlatih YOLOv8n-seg
import os
import glob
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display

# ======================================================================
# 8.1 Simulasi BatchNorm2d dari Nol (NumPy) + Histogram Distribusi
# ======================================================================
rng = np.random.default_rng(42)
# Simulasi feature map [Batch=4, Channel=3, H=16, W=16] sebelum BatchNorm
x_raw = np.stack([
    rng.normal(loc=5.0, scale=3.5, size=(4, 16, 16)),   # Kanal 0: rata-rata tinggi, sebaran lebar
    rng.normal(loc=-2.5, scale=0.8, size=(4, 16, 16)),  # Kanal 1: rata-rata negatif, sebaran sempit
    rng.normal(loc=12.0, scale=6.0, size=(4, 16, 16))   # Kanal 2: skala besar
], axis=1)

eps = 1e-5
mean_c = x_raw.mean(axis=(0, 2, 3), keepdims=True)
var_c = x_raw.var(axis=(0, 2, 3), keepdims=True)
x_norm = (x_raw - mean_c) / np.sqrt(var_c + eps)

gamma = np.array([1.5, 0.7, 2.0]).reshape(1, 3, 1, 1)
beta = np.array([0.2, -0.3, 0.5]).reshape(1, 3, 1, 1)
x_bn = gamma * x_norm + beta

tabel_bn_sim = pd.DataFrame({
    'Kanal': [0, 1, 2],
    'Mean Sebelum': x_raw.mean(axis=(0, 2, 3)).round(4),
    'Std Sebelum': x_raw.std(axis=(0, 2, 3)).round(4),
    'Mean Sesudah Norm (x_hat)': x_norm.mean(axis=(0, 2, 3)).round(4),
    'Std Sesudah Norm (x_hat)': x_norm.std(axis=(0, 2, 3)).round(4),
    'Gamma (scale)': gamma.flatten(),
    'Beta (shift)': beta.flatten(),
    'Mean Akhir (y)': x_bn.mean(axis=(0, 2, 3)).round(4),
    'Std Akhir (y)': x_bn.std(axis=(0, 2, 3)).round(4)
})
print('TABEL 5. Statistik BatchNorm2d per Kanal (Simulasi NumPy)')
display(tabel_bn_sim)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for c in range(3):
    axes[0].hist(x_raw[:, c].ravel(), bins=30, alpha=0.6, label=f'Kanal {c}')
    axes[1].hist(x_norm[:, c].ravel(), bins=30, alpha=0.6, label=f'Kanal {c}')
    axes[2].hist(x_bn[:, c].ravel(), bins=30, alpha=0.6, label=f'Kanal {c}')
axes[0].set_title('1. Sebelum BatchNorm (Raw Conv)')
axes[1].set_title('2. Sesudah Normalisasi (Mean=0, Std=1)')
axes[2].set_title('3. Sesudah Skala Gamma & Geser Beta')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/grafik_batchnorm_tahapan.png', dpi=200)
plt.show()

# ======================================================================
# 8.2 Grafik Fungsi Aktivasi SiLU vs ReLU & Turunannya
# ======================================================================
x_vals = np.linspace(-5, 5, 500)
sigmoid_x = 1 / (1 + np.exp(-x_vals))
silu_x = x_vals * sigmoid_x
dsilu_x = sigmoid_x * (1 + x_vals * (1 - sigmoid_x))

relu_x = np.maximum(0, x_vals)
drelu_x = (x_vals > 0).astype(float)

titik_uji = np.array([-3.0, -1.28, -0.5, 0.0, 0.5, 1.0, 3.0])
sig_t = 1 / (1 + np.exp(-titik_uji))
tabel_aktivasi = pd.DataFrame({
    'x': titik_uji,
    'SiLU(x)': np.round(titik_uji * sig_t, 4),
    'Turunan SiLU': np.round(sig_t * (1 + titik_uji * (1 - sig_t)), 4),
    'ReLU(x)': np.maximum(0, titik_uji),
    'Turunan ReLU': (titik_uji > 0).astype(float)
})
print('\nTABEL 6. Perbandingan Nilai Fungsi & Turunan SiLU vs ReLU')
display(tabel_aktivasi)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(x_vals, silu_x, label='SiLU (Swish): x * sigmoid(x)', color='#2563eb', linewidth=2.2)
ax1.plot(x_vals, relu_x, label='ReLU: max(0, x)', color='#dc2626', linestyle='--', linewidth=2)
ax1.scatter([-1.2785], [-0.2785], color='#1d4ed8', zorder=5, label='Min SiLU (-1.28, -0.2785)')
ax1.axhline(0, color='gray', linewidth=0.8)
ax1.axvline(0, color='gray', linewidth=0.8)
ax1.set_title('Kurva Fungsi Aktivasi SiLU vs ReLU')
ax1.set_xlabel('Input (x)')
ax1.set_ylabel('Output Aktivasi')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(x_vals, dsilu_x, label="Turunan SiLU'(x)", color='#2563eb', linewidth=2.2)
ax2.plot(x_vals, drelu_x, label="Turunan ReLU'(x)", color='#dc2626', linestyle='--', linewidth=2)
ax2.axhline(0, color='gray', linewidth=0.8)
ax2.axvline(0, color='gray', linewidth=0.8)
ax2.set_title('Kurva Gradien / Turunan Pertama (Backpropagation)')
ax2.set_xlabel('Input (x)')
ax2.set_ylabel('Gradien')
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/grafik_aktivasi_silu_relu.png', dpi=200)
plt.show()

# ======================================================================
# 8.3 & 8.4 Inspeksi BatchNorm2d, SiLU vs ReLU & Efek Fuse pada YOLOv8n-seg
# ======================================================================
gammas, betas, r_means, r_vars = [], [], [], []
for m in net.modules():
    if isinstance(m, nn.BatchNorm2d):
        gammas.append(m.weight.detach().cpu().numpy())
        betas.append(m.bias.detach().cpu().numpy())
        r_means.append(m.running_mean.detach().cpu().numpy())
        r_vars.append(m.running_var.detach().cpu().numpy())

all_gamma = np.concatenate(gammas)
all_beta = np.concatenate(betas)
all_rmean = np.concatenate(r_means)
all_rvar = np.concatenate(r_vars)

fig, axs = plt.subplots(2, 2, figsize=(11, 6.5))
axs[0, 0].hist(all_gamma, bins=40, color='#2563eb', alpha=0.85)
axs[0, 0].set_title(f'Sebaran Gamma (Weight) BatchNorm2d (n={len(all_gamma)})')
axs[0, 1].hist(all_beta, bins=40, color='#16a34a', alpha=0.85)
axs[0, 1].set_title(f'Sebaran Beta (Bias) BatchNorm2d (n={len(all_beta)})')
axs[1, 0].hist(all_rmean, bins=40, color='#ea580c', alpha=0.85)
axs[1, 0].set_title('Sebaran Running Mean BatchNorm2d')
axs[1, 1].hist(all_rvar, bins=40, color='#9333ea', alpha=0.85)
axs[1, 1].set_title('Sebaran Running Variance BatchNorm2d')
for ax in axs.ravel():
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/grafik_sebaran_batchnorm_model.png', dpi=200)
plt.show()

# Uji Layer Pertama (Conv + BN + SiLU vs ReLU) pada Sampel Gambar Telur
sample_imgs = glob.glob('/content/dataset/**/images/*.jpg', recursive=True) + glob.glob('/content/dataset/**/images/*.png', recursive=True)
if sample_imgs:
    img_pil = PILImage.open(sample_imgs[0]).convert('RGB').resize((640, 640))
    img_arr = np.asarray(img_pil, dtype=np.float32) / 255.0
    img_tensor = torch.from_numpy(img_arr).permute(2, 0, 1).unsqueeze(0)
    sumber_gambar = os.path.basename(sample_imgs[0])
else:
    img_tensor = torch.rand(1, 3, 640, 640)
    sumber_gambar = 'Citra Sintetis 640x640 (dataset belum diekstrak)'

first_block = net.model[0]  # Conv(3 -> 16, k=3, s=2)
with torch.no_grad():
    after_bn = first_block.bn(first_block.conv(img_tensor))
    out_silu = nn.SiLU()(after_bn).numpy()
    out_relu = nn.ReLU()(after_bn).numpy()

tabel_uji_gambar = pd.DataFrame([
    {
        'Aktivasi (Layer 0)': 'SiLU (Bawaan YOLOv8-seg)',
        'Sampel Uji': sumber_gambar,
        'Persen Nilai Nol Tepat (%)': round(float((out_silu == 0).mean() * 100), 2),
        'Nilai Minimum': round(float(out_silu.min()), 4),
        'Nilai Maksimum': round(float(out_silu.max()), 4),
        'Rata-rata Aktivasi': round(float(out_silu.mean()), 4)
    },
    {
        'Aktivasi (Layer 0)': 'ReLU (Pembanding)',
        'Sampel Uji': sumber_gambar,
        'Persen Nilai Nol Tepat (%)': round(float((out_relu == 0).mean() * 100), 2),
        'Nilai Minimum': round(float(out_relu.min()), 4),
        'Nilai Maksimum': round(float(out_relu.max()), 4),
        'Rata-rata Aktivasi': round(float(out_relu.mean()), 4)
    }
])
print('\nTABEL 7. Perbandingan Aktivasi SiLU vs ReLU pada Layer Pertama YOLOv8n-seg')
display(tabel_uji_gambar)

# Cek Efek Fuse (Conv2d + BatchNorm2d digabung saat inferensi)
net_fused = copy.deepcopy(net).fuse()
tabel_fuse = pd.DataFrame([
    {
        'Kondisi Model': 'Sebelum Fuse (Saat Training)',
        'Jumlah Conv2d': sum(1 for m in net.modules() if isinstance(m, nn.Conv2d)),
        'Jumlah BatchNorm2d': sum(1 for m in net.modules() if isinstance(m, nn.BatchNorm2d)),
        'Total Parameter': sum(p.numel() for p in net.parameters())
    },
    {
        'Kondisi Model': 'Sesudah Fuse (Saat Predict / Export)',
        'Jumlah Conv2d': sum(1 for m in net_fused.modules() if isinstance(m, nn.Conv2d)),
        'Jumlah BatchNorm2d': sum(1 for m in net_fused.modules() if isinstance(m, nn.BatchNorm2d)),
        'Total Parameter': sum(p.numel() for p in net_fused.parameters())
    }
])
print('\nTABEL 8. Efek Penggabungan Layer (Model Fuse) pada YOLOv8n-seg')
display(tabel_fuse)

tabel_bn_sim.to_csv('/content/info_batchnorm_simulasi.csv', index=False)
tabel_aktivasi.to_csv('/content/info_aktivasi_silu_relu.csv', index=False)
tabel_uji_gambar.to_csv('/content/info_uji_silu_relu_model.csv', index=False)
tabel_fuse.to_csv('/content/info_efek_fuse_model.csv', index=False)
print('Tersimpan semua CSV analisis BatchNorm & Aktivasi!')

In [ ]:
# 9. Unduh Bobot Model Segmentasi Terbaik (best.pt)
from google.colab import files

best_seg_weight = '/content/runs/egg_defect_yolov8_seg/weights/best.pt'
if os.path.exists(best_seg_weight):
    print('⬇️ Mengunduh file best.pt ke komputer Anda...')
    files.download(best_seg_weight)
else:
    print('⚠️ File best.pt belum ditemukan. Pastikan proses training sudah selesai.')

In [ ]:
# 10. Unduh Seluruh Grafik Pelatihan, Matriks Konfusi, Tabel CSV & Struktur Model (.ZIP)
import os
import shutil
import glob
from IPython.display import Image, display
from google.colab import files

# 1. Deteksi otomatis folder hasil pelatihan YOLOv8-seg
run_dir = None
if 'results' in locals() and hasattr(results, 'save_dir'):
    run_dir = str(results.save_dir)
elif os.path.exists('/content/runs/egg_defect_yolov8_seg'):
    run_dir = '/content/runs/egg_defect_yolov8_seg'
else:
    candidates = glob.glob('/content/runs/**/results.png', recursive=True)
    if candidates:
        run_dir = os.path.dirname(candidates[0])

export_dir = '/content/grafik_dan_analisis_yolov8seg'
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)
os.makedirs(export_dir, exist_ok=True)

copied_count = 0
plot_extensions = ('.png', '.jpg', '.jpeg', '.csv', '.txt')

# Salin grafik & CSV dari folder training (jika sudah training)
if run_dir and os.path.exists(run_dir):
    print(f'📂 Folder hasil training ditemukan: {run_dir}')
    cm_path = os.path.join(run_dir, 'confusion_matrix_normalized.png')
    if not os.path.exists(cm_path):
        cm_path = os.path.join(run_dir, 'confusion_matrix.png')
    res_plot_path = os.path.join(run_dir, 'results.png')
    if os.path.exists(cm_path):
        print('\n📊 Preview Matriks Konfusi (Normalized):')
        display(Image(filename=cm_path, width=600))
    if os.path.exists(res_plot_path):
        print('\n📈 Preview Kurva Loss dan mAP Pelatihan:')
        display(Image(filename=res_plot_path, width=800))
    for fname in os.listdir(run_dir):
        if fname.lower().endswith(plot_extensions):
            shutil.copy2(os.path.join(run_dir, fname), os.path.join(export_dir, fname))
            copied_count += 1

# Salin seluruh tabel CSV, struktur_model.txt, dan grafik analisis arsitektur dari /content
extra_files = (
    glob.glob('/content/info_*.csv') +
    glob.glob('/content/grafik_*.png') +
    (['/content/struktur_model.txt'] if os.path.exists('/content/struktur_model.txt') else [])
)
for fpath in extra_files:
    shutil.copy2(fpath, os.path.join(export_dir, os.path.basename(fpath)))
    copied_count += 1

zip_output_path = '/content/grafik_dan_analisis_yolov8seg.zip'
if os.path.exists(zip_output_path):
    os.remove(zip_output_path)

shutil.make_archive('/content/grafik_dan_analisis_yolov8seg', 'zip', export_dir)
print(f'\n📦 Berhasil mengemas {copied_count} file grafik, kurva, matriks, tabel CSV & struktur pohon!')
print("⬇️ Memulai unduhan otomatis: 'grafik_dan_analisis_yolov8seg.zip'...")
files.download(zip_output_path)